# Exploración del catálogo Data360 — prototipo

Prototipo de las **3 visualizaciones de exploración** elegidas en `docs/plans/01-tp-final-plan.md` §4.5:

1. **Top-N bases por cantidad de indicadores** (panorama del catálogo).
2. **Dumbbell fin nominal vs. último dato real** (calidad / frescura).
3. **Scatter indicadores vs. cobertura mediana** (amplitud vs. profundidad, resaltando el shortlist temático).

Lee únicamente los CSV de `data/derived/`; no vuelve a consultar la API. La versión final (Altair exportado a HTML) se hará después de validar este prototipo.

In [1]:
from pathlib import Path

import altair as alt
import pandas as pd

pd.set_option("display.max_columns", 50)
alt.data_transformers.enable("default")


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "derived" / "database_summary.csv").exists():
            return candidate
    raise FileNotFoundError("No se encontró data/derived/database_summary.csv")


ROOT = find_project_root()
DERIVED = ROOT / "data" / "derived"
print("Proyecto:", ROOT)
print("CSV:", sorted(p.name for p in DERIVED.glob("*.csv")))

Proyecto: /Users/selewaut/Code Projects/master/world_bank_infoviz
CSV: ['database_actual_freshness.csv', 'database_doc_counts.csv', 'database_region_coverage.csv', 'database_summary.csv', 'database_topics.csv', 'freshness_summary.csv', 'granular_freshness.csv', 'indicators.csv', 'region_overall.csv', 'region_summary.csv', 'topic_summary.csv']


## 1. Top-N bases por cantidad de indicadores

**Pregunta:** ¿qué bases concentran el catálogo?  
**Codificación:** barras horizontales, `x = indicadores`, `y = base`, `color = tema L1 dominante` (`top_l1_topics`, primer tema).  
**Fuente:** `data/derived/database_summary.csv`.

In [2]:
databases = pd.read_csv(DERIVED / "database_summary.csv")


def dominant_l1(value: str) -> str:
    if not isinstance(value, str) or not value.strip():
        return "Sin clasificar"
    return value.split("|")[0].split(":")[0].strip()


databases["dominant_l1"] = databases["top_l1_topics"].map(dominant_l1)
top20 = databases.sort_values("indicators", ascending=False).head(20).copy()

total = databases["indicators"].sum()
share = top20["indicators"].sum() / total
print(f"Bases: {len(databases):,} · indicadores: {total:,}")
print(f"Top 20 = {top20['indicators'].sum():,} indicadores ({share:.1%} del total)")
print(top20[["database_id", "indicators", "coverage_median", "dominant_l1"]].to_string(index=False))

bars = (
    alt.Chart(top20)
    .mark_bar(cornerRadiusEnd=3)
    .encode(
        x=alt.X("indicators:Q", title="Cantidad de indicadores"),
        y=alt.Y("database_id:N", sort="-x", title=None),
        color=alt.Color("dominant_l1:N", title="Tema L1 dominante"),
        tooltip=[
            alt.Tooltip("database_id:N", title="Base"),
            alt.Tooltip("database_name:N", title="Nombre"),
            alt.Tooltip("indicators:Q", title="Indicadores", format=","),
            alt.Tooltip("coverage_median:Q", title="Cobertura mediana"),
            alt.Tooltip("dominant_l1:N", title="Tema L1"),
        ],
    )
    .properties(width=660, height=460, title="Top 20 bases por cantidad de indicadores")
)

labels = bars.mark_text(align="left", dx=4, fontSize=10).encode(
    text=alt.Text("indicators:Q", format=","), color=alt.value("#333333")
)

(bars + labels).configure_axis(grid=True, gridOpacity=0.2)

Bases: 171 · indicadores: 9,859
Top 20 = 7,447 indicadores (75.5% del total)
database_id  indicators  coverage_median                           dominant_l1
     WB_WDI        1545            183.0                             Education
    IMF_BOP        1146             50.0                       Economic Policy
 WB_EDSTATS        1072             96.0                             Education
      WB_ES         584            180.0 Trade, Investment and Competitiveness
    IMF_IFS         495             23.0                               Finance
      WB_GS         363            217.0                                Gender
  WB_FINDEX         280             90.0                               Finance
IMF_GFS_SOO         218            151.0                       Economic Policy
     BS_SGI         193             41.0                          Institutions
     WB_HNP         192            217.0        Health, Nutrition & Population
   IMF_FSIC         184             62.0              

alt.LayerChart(...)

## 2. Dumbbell: fin nominal vs. último dato real

**Pregunta:** ¿la metadata promete datos más recientes de los que realmente existen?  
**Codificación:** un par de puntos por base (fin nominal y último dato observado), unidos por una línea; se muestran las 30 bases con mayor brecha absoluta.  
**Fuente:** `data/derived/database_actual_freshness.csv`.

- `gap = nominal_end − actual_latest_year`.
- `gap > 0` → la metadata declara un fin posterior al último dato real (desactualización / proyección).
- `gap < 0` → hay datos más allá del fin declarado (metadata conservadora).

In [3]:
freshness = pd.read_csv(DERIVED / "database_actual_freshness.csv")
freshness["actual_year"] = freshness["actual_latest_year"].astype(str).str.extract(r"(\d{4})").astype(int)
freshness["nominal_year"] = freshness["nominal_end"].astype(str).str.extract(r"(\d{4})").astype(int)
freshness["gap"] = freshness["nominal_year"] - freshness["actual_year"]
freshness["estado"] = freshness["gap"].map(
    lambda g: "Metadata sobreestima" if g > 0 else ("Metadata subestima" if g < 0 else "Coincide")
)

print("Bases sondeadas:", len(freshness))
print(freshness["estado"].value_counts().to_string())
print("\nMayores brechas:")
print(
    freshness.reindex(freshness["gap"].abs().sort_values(ascending=False).index)
    .head(15)[["database_id", "actual_latest_period", "nominal_end", "gap", "estado"]]
    .to_string(index=False)
)

shown = freshness.reindex(freshness["gap"].abs().sort_values(ascending=False).index).head(30)
order = shown.sort_values("gap")["database_id"].tolist()

long = shown.melt(
    id_vars=["database_id", "gap", "estado"],
    value_vars=["actual_year", "nominal_year"],
    var_name="serie",
    value_name="year",
)
long["serie"] = long["serie"].map(
    {"actual_year": "Último dato real", "nominal_year": "Fin nominal (metadata)"}
)

rule = (
    alt.Chart(shown)
    .mark_rule(color="#b0b0b0", strokeWidth=2)
    .encode(
        y=alt.Y("database_id:N", sort=order, title=None),
        x=alt.X("actual_year:Q", title="Año", scale=alt.Scale(zero=False)),
        x2="nominal_year:Q",
    )
)

points = (
    alt.Chart(long)
    .mark_circle(size=90, opacity=0.9)
    .encode(
        y=alt.Y("database_id:N", sort=order, title=None),
        x=alt.X("year:Q", title="Año", scale=alt.Scale(zero=False)),
        color=alt.Color(
            "serie:N",
            title=None,
            scale=alt.Scale(
                domain=["Último dato real", "Fin nominal (metadata)"],
                range=["#1f77b4", "#d62728"],
            ),
        ),
        tooltip=[
            alt.Tooltip("database_id:N", title="Base"),
            alt.Tooltip("serie:N", title="Serie"),
            alt.Tooltip("year:Q", title="Año"),
            alt.Tooltip("gap:Q", title="Brecha (años)"),
            alt.Tooltip("estado:N", title="Estado"),
        ],
    )
)

(rule + points).properties(
    width=620, height=640, title="Fin nominal vs. último dato real — 30 mayores brechas"
)

Bases sondeadas: 158
estado
Coincide                114
Metadata sobreestima     39
Metadata subestima        5

Mayores brechas:
 database_id actual_latest_period nominal_end  gap               estado
IMF_GFS_SSUC                 2014        2025   11 Metadata sobreestima
IMF_GFS_SFCP                 2014        2025   11 Metadata sobreestima
  IMF_GFS_BS                 2014        2025   11 Metadata sobreestima
      WB_PIP                 2014        2025   11 Metadata sobreestima
    WB_BOOST                 2020        2030   10 Metadata sobreestima
    IMF_QGFS              2016-Q2     2026-Q2   10 Metadata sobreestima
     WB_LPGD                 2013        2023   10 Metadata sobreestima
   IMF_FSIRE                 2014        2023    9 Metadata sobreestima
  UNESCO_UIS                 2018        2026    8 Metadata sobreestima
      EPC_AI                 2018        2025    7 Metadata sobreestima
      WB_SHP                 2017        2022    5 Metadata sobreestima
     I

alt.LayerChart(...)

## 3. Scatter amplitud vs. profundidad

**Pregunta:** ¿las bases con más indicadores también cubren más economías?  
**Codificación:** `x = indicadores` (escala log), `y = cobertura mediana`, `tamaño = cobertura máxima`, color = pertenencia al shortlist temático.  
**Fuente:** `data/derived/database_summary.csv`.

El shortlist son las 8 bases del recorte datacenter / conectividad / energía / infraestructura (`PEERING_DB`, `PCH_IXP`, `OOKLA_ST`, `ITU_DH`, `WB_WDI`, `WB_SE4ALL`, `WB_RISE`, `WB_PPI`).

In [4]:
SHORTLIST = {
    "PEERING_DB": "Datacenters",
    "PCH_IXP": "IXPs",
    "OOKLA_ST": "Velocidad",
    "ITU_DH": "Conectividad",
    "WB_WDI": "Internet / energía",
    "WB_SE4ALL": "Energía",
    "WB_RISE": "Energía (política)",
    "WB_PPI": "Inversión infra",
}

databases["grupo"] = databases["database_id"].map(
    lambda db: "Shortlist temático" if db in SHORTLIST else "Resto del catálogo"
)
databases["shortlist_role"] = databases["database_id"].map(SHORTLIST).fillna("")

print(databases[databases["database_id"].isin(SHORTLIST)][
    ["database_id", "shortlist_role", "indicators", "coverage_median", "coverage_max"]
].sort_values("indicators", ascending=False).to_string(index=False))

base = (
    alt.Chart(databases)
    .mark_circle(opacity=0.65, stroke="white", strokeWidth=0.5)
    .encode(
        x=alt.X(
            "indicators:Q",
            title="Cantidad de indicadores (escala log)",
            scale=alt.Scale(type="log"),
        ),
        y=alt.Y("coverage_median:Q", title="Cobertura mediana (economías)"),
        size=alt.Size(
            "coverage_max:Q",
            title="Cobertura máxima",
            scale=alt.Scale(range=[20, 450]),
        ),
        color=alt.Color(
            "grupo:N",
            title=None,
            scale=alt.Scale(
                domain=["Shortlist temático", "Resto del catálogo"],
                range=["#d62728", "#9aa5b1"],
            ),
        ),
        tooltip=[
            alt.Tooltip("database_id:N", title="Base"),
            alt.Tooltip("shortlist_role:N", title="Rol en el shortlist"),
            alt.Tooltip("indicators:Q", title="Indicadores", format=","),
            alt.Tooltip("coverage_median:Q", title="Cobertura mediana"),
            alt.Tooltip("coverage_max:Q", title="Cobertura máxima"),
        ],
    )
)

labels = (
    alt.Chart(databases[databases["database_id"].isin(SHORTLIST)])
    .mark_text(dx=7, dy=-5, fontSize=10, fontWeight="bold")
    .encode(
        x="indicators:Q",
        y="coverage_median:Q",
        text="database_id:N",
        color=alt.value("#a4161a"),
    )
)

(base + labels).properties(
    width=660, height=440, title="Amplitud vs. profundidad del catálogo (shortlist resaltado)"
).interactive()

database_id     shortlist_role  indicators  coverage_median  coverage_max
     WB_WDI Internet / energía        1545            183.0           217
     ITU_DH       Conectividad          39            189.0           230
    WB_RISE Energía (política)          27            140.0           140
  WB_SE4ALL            Energía           8            219.5           228
     WB_PPI    Inversión infra           5            112.0           133
   OOKLA_ST          Velocidad           2            242.0           244
    PCH_IXP               IXPs           2            156.0           156
 PEERING_DB        Datacenters           1            158.0           158


alt.LayerChart(...)

## Notas para la versión final

- Las tres figuras cubren ángulos distintos: **panorama (#2)**, **calidad (#4)** y **selección (#6)**.
- `#4` concentra el hallazgo más fuerte: 44 de 158 bases no coinciden entre fin nominal y último dato real.
- Antes de exportar: fijar tipografía/formato, títulos en lenguaje natural y nota de fuente al pie.
- Las figuras descartadas (`#1`, `#3`, `#5`) quedan justificadas en `docs/plans/01-tp-final-plan.md` §4.5.